# LitSpace Evaluation Analysis

This notebook goes through the committed evaluation results for LitSpace. It loads the saved metrics, regenerates the plots used in the README, and explains what the numbers say about the current version of the system.

The point is not just to show the best-looking results. It is also to be clear about where LitSpace is working well, where it is only partly working, and where the same weak spots keep showing up.


## Notebook Roadmap

The notebook follows a simple order:

1. Set up paths and load the saved results.
2. Look at what is actually in the benchmark.
3. Compare LitSpace to the two baselines overall.
4. Separate retrieval performance from answer quality.
5. Look at pairwise wins, category-level results, and failure types.
6. End with the quick demo path.

## Useful Commands

Use these commands from the repository root.

```bash
# Open the notebook
jupyter notebook evaluation/notebooks/evaluation_analysis.ipynb

# Quick demo: create a demo project and run 3 sample questions
backend/.litenv/bin/python demo/quick_demo.py

# Static fallback from committed outputs
backend/.litenv/bin/python demo/quick_demo.py --static

# Full evaluation pipeline
backend/.litenv/bin/python evaluation/scripts/setup_project.py
backend/.litenv/bin/python evaluation/scripts/run_systems.py
backend/.litenv/bin/python evaluation/scripts/judge_answers.py
backend/.litenv/bin/python evaluation/scripts/pairwise_judge.py
backend/.litenv/bin/python evaluation/scripts/summarize_results.py
```


## 1. Setup and Paths

This part imports the standard-library modules used in the notebook and sets the main paths. The path logic is written so the notebook still works whether it is opened from the repo root or from inside `evaluation/notebooks/`.

Main files used here:

- `evaluation/results/metrics_summary.json` for the aggregated metrics
- `evaluation/results/error_analysis.csv` for row-level failure labels
- `evaluation/datasets/questions.csv` for benchmark composition
- `evaluation/results/plots/` for the saved figures this notebook regenerates


In [47]:
from __future__ import annotations

import csv
import json
from collections import Counter
from html import escape
from pathlib import Path

from IPython.display import Markdown, display

# Resolve the repo root once so the rest of the paths are stable.
# This works from the repo root and from evaluation/notebooks.
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
EVAL_DIR = ROOT / 'evaluation'
RESULTS_DIR = EVAL_DIR / 'results'
PLOTS_DIR = RESULTS_DIR / 'plots'
SUMMARY_JSON = RESULTS_DIR / 'metrics_summary.json'
ERROR_CSV = RESULTS_DIR / 'error_analysis.csv'
QUESTIONS_CSV = EVAL_DIR / 'datasets' / 'questions.csv'

print('Root:', ROOT)
print('Summary file:', SUMMARY_JSON)
print('Error file:', ERROR_CSV)
print('Questions file:', QUESTIONS_CSV)
print('Plots dir:', PLOTS_DIR)


def markdown_table(headers: list[str], rows: list[list[str]]) -> str:
    header = '| ' + ' | '.join(headers) + ' |'
    divider = '| ' + ' | '.join('---' if index == 0 else '---:' for index, _ in enumerate(headers)) + ' |'
    body = ['| ' + ' | '.join(row) + ' |' for row in rows]
    return '\n'.join([header, divider, *body])


def display_markdown_table(headers: list[str], rows: list[list[str]], extra: str = '') -> None:
    content = markdown_table(headers, rows)
    if extra:
        content += '\n\n' + extra
    display(Markdown(content))


Root: /Users/enasbatarfi/DS593-LLM/litspace
Summary file: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/metrics_summary.json
Error file: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/error_analysis.csv
Questions file: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/datasets/questions.csv
Plots dir: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots


## 2. Load the Main Results

The summary file already contains the main outputs from the evaluation pipeline. It includes:

- overall averages for each system
- pairwise comparison rates
- category-level breakdowns

Loading them once here keeps the rest of the notebook focused on the results instead of file handling.


In [48]:
# Load the saved metrics used throughout the notebook.
metrics = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
overall = metrics['overall']
pairwise = metrics['pairwise']
by_category = metrics['by_category']

# Make sure the plot directory exists before saving refreshed SVGs.
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Systems found:', ', '.join(overall.keys()))
print('Pairwise sets:', ', '.join(pairwise.keys()))
print('Categories:', ', '.join(by_category.keys()))


Systems found: litspace_rag, zero_shot, summary_few_shot
Pairwise sets: litspace_vs_zero_shot, litspace_vs_summary_few_shot
Categories: ambiguity, evidence_lookup, followup, multi_paper_synthesis, refusal, single_paper_factual, single_paper_summary


## 3. Benchmark Overview

Before looking at the model scores, it helps to look at the benchmark itself. The evaluation set has 45 questions spread across seven categories: factual questions, summaries, multi-paper synthesis, evidence lookup, follow-up behavior, ambiguity handling, and refusal behavior.

That matters because LitSpace is trying to do more than one thing. A system can do well on direct factual questions and still struggle when it has to combine papers or decide whether it should clarify first.


In [49]:
# Count examples in each benchmark category.
# This helps make sense of the category results later.
category_counts = Counter()
with QUESTIONS_CSV.open('r', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        category_counts[row['category']] += 1

category_order = [
    'single_paper_factual',
    'single_paper_summary',
    'multi_paper_synthesis',
    'evidence_lookup',
    'followup',
    'ambiguity',
    'refusal',
]

benchmark_rows = [(category, category_counts[category]) for category in category_order]
benchmark_rows

CATEGORY_LABELS = {
    'single_paper_factual': 'Single-paper factual',
    'single_paper_summary': 'Single-paper summary',
    'multi_paper_synthesis': 'Multi-paper synthesis',
    'evidence_lookup': 'Evidence lookup',
    'followup': 'Follow-up',
    'ambiguity': 'Ambiguity',
    'refusal': 'Refusal',
}


In [50]:
# Render the benchmark category table from evaluation/datasets/questions.csv.
display_markdown_table(
    ['Category', 'Questions'],
    [[CATEGORY_LABELS.get(category, category.replace('_', ' ').title()), str(count)] for category, count in benchmark_rows],
)


| Category | Questions |
| --- | ---: |
| Single-paper factual | 7 |
| Single-paper summary | 7 |
| Multi-paper synthesis | 8 |
| Evidence lookup | 7 |
| Follow-up | 7 |
| Ambiguity | 5 |
| Refusal | 4 |

## 4. Overall System Comparison

This is the main baseline check: how does LitSpace compare to the two non-retrieval baselines on overall answer quality?

A few things matter when reading this table:

- correctness, completeness, relevance, helpfulness, and faithfulness are judge-based scores
- correctness is on a 0 to 2 scale
- the point is not only whether LitSpace wins, but also where the stronger baseline stays competitive


In [51]:
# Keep shared labels and colors in one place.
SYSTEM_LABELS = {
    'litspace_rag': 'LitSpace RAG',
    'zero_shot': 'Zero-shot',
    'summary_few_shot': 'Summary few-shot',
}

SYSTEM_COLORS = {
    'litspace_rag': '#0f766e',
    'zero_shot': '#c2410c',
    'summary_few_shot': '#1d4ed8',
}

def fmt_num(value, digits=2):
    return 'N/A' if value is None else f'{value:.{digits}f}'

def fmt_pct(value, digits=1):
    return 'N/A' if value is None else f'{value * 100:.{digits}f}%'

def fmt_cost(value):
    return 'N/A' if value is None else f'${value:.5f}'




def svg_document(width: int, height: int, body: str) -> str:
    """Wrap SVG body content with the shared notebook chart style."""
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" role="img">'
        '<style>'
        '.title{font:700 24px Arial,sans-serif;fill:#0f172a;}'
        '.subtitle{font:400 13px Arial,sans-serif;fill:#475569;}'
        '.label{font:600 13px Arial,sans-serif;fill:#0f172a;}'
        '.tick{font:400 12px Arial,sans-serif;fill:#64748b;}'
        '.value{font:700 12px Arial,sans-serif;fill:#0f172a;}'
        '.foot{font:400 11px Arial,sans-serif;fill:#64748b;}'
        '</style>'
        f'{body}</svg>'
    )


def horizontal_bar_chart(*, title: str, subtitle: str, items: list[dict], output_path: Path, max_value: float, value_fmt: str, footnote: str) -> None:
    """Save a horizontal bar chart as SVG."""
    width = 960
    top = 90
    left = 280
    right = 80
    row_h = 52
    chart_h = row_h * len(items)
    height = top + chart_h + 70
    chart_w = width - left - right

    # Add reference lines so the bar lengths are easier to compare.
    grid_values = [0, max_value * 0.25, max_value * 0.5, max_value * 0.75, max_value]
    body = [
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#f8fafc" />',
        f'<text x="36" y="42" class="title">{escape(title)}</text>',
        f'<text x="36" y="64" class="subtitle">{escape(subtitle)}</text>',
    ]

    for grid_value in grid_values:
        x = left + (grid_value / max_value) * chart_w if max_value else left
        label = value_fmt.format(grid_value)
        body.append(f'<line x1="{x:.2f}" y1="{top - 10}" x2="{x:.2f}" y2="{top + chart_h}" stroke="#e2e8f0" stroke-width="1" />')
        body.append(f'<text x="{x:.2f}" y="{top + chart_h + 20}" class="tick" text-anchor="middle">{escape(label)}</text>')

    for index, item in enumerate(items):
        y = top + index * row_h
        bar_y = y + 8
        bar_h = 24
        bar_w = (item['value'] / max_value) * chart_w if max_value else 0
        value_text = value_fmt.format(item['value'])
        body.append(f'<text x="{left - 16}" y="{bar_y + 16}" class="label" text-anchor="end">{escape(item["label"])}</text>')
        body.append(f'<rect x="{left}" y="{bar_y}" width="{chart_w}" height="{bar_h}" rx="8" fill="#e2e8f0" />')
        body.append(f'<rect x="{left}" y="{bar_y}" width="{bar_w:.2f}" height="{bar_h}" rx="8" fill="{item["color"]}" />')
        body.append(f'<text x="{left + bar_w + 10:.2f}" y="{bar_y + 16}" class="value">{escape(value_text)}</text>')

    body.append(f'<text x="36" y="{height - 22}" class="foot">{escape(footnote)}</text>')
    output_path.write_text(svg_document(width, height, ''.join(body)), encoding='utf-8')


In [52]:
# Save the overall comparison plot used in the README.
overall_plot = PLOTS_DIR / 'overall_correctness.svg'

horizontal_bar_chart(
    title='Overall Correctness Mean',
    subtitle='Higher is better. Values come from evaluation/results/metrics_summary.json.',
    items=[
        {'label': SYSTEM_LABELS['litspace_rag'], 'value': overall['litspace_rag']['correctness_mean'], 'color': SYSTEM_COLORS['litspace_rag']},
        {'label': SYSTEM_LABELS['summary_few_shot'], 'value': overall['summary_few_shot']['correctness_mean'], 'color': SYSTEM_COLORS['summary_few_shot']},
        {'label': SYSTEM_LABELS['zero_shot'], 'value': overall['zero_shot']['correctness_mean'], 'color': SYSTEM_COLORS['zero_shot']},
    ],
    output_path=overall_plot,
    max_value=2.0,
    value_fmt='{:.2f}',
    footnote='Judge score range: 0 to 2.',
)

print('Saved plot:', overall_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/overall_correctness.svg


In [53]:
# Render the overall comparison table from metrics_summary.json.
overall_rows = []
for system in ['litspace_rag', 'summary_few_shot', 'zero_shot']:
    values = overall[system]
    overall_rows.append([
        SYSTEM_LABELS[system],
        fmt_num(values.get('correctness_mean')),
        fmt_num(values.get('completeness_mean')),
        fmt_num(values.get('relevance_mean')),
        fmt_num(values.get('helpfulness_mean')),
        fmt_num(values.get('faithfulness_mean')),
        fmt_num(values.get('avg_latency_sec')),
    ])

display_markdown_table(
    ['System', 'Correctness', 'Completeness', 'Relevance', 'Helpfulness', 'Faithfulness', 'Avg latency (s)'],
    overall_rows,
    '![Overall correctness](../results/plots/overall_correctness.svg)\n\nThis table shows LitSpace clearly beating zero-shot. The gap is large on correctness, completeness, and helpfulness, so retrieval is making a real difference here.\n\nThe comparison with summary few-shot is closer. LitSpace is a bit better on correctness, completeness, and helpfulness, while summary few-shot is stronger on relevance and faithfulness. A fair read is that LitSpace is better overall, but not by a huge margin on every metric.',
)


| System | Correctness | Completeness | Relevance | Helpfulness | Faithfulness | Avg latency (s) |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| LitSpace RAG | 1.42 | 1.27 | 1.71 | 1.42 | 1.44 | 2.23 |
| Summary few-shot | 1.33 | 1.02 | 1.82 | 1.27 | 1.82 | 2.08 |
| Zero-shot | 0.33 | 0.22 | 1.22 | 0.31 | N/A | 2.04 |

![Overall correctness](../results/plots/overall_correctness.svg)

This table shows LitSpace clearly beating zero-shot. The gap is large on correctness, completeness, and helpfulness, so retrieval is making a real difference here.

The comparison with summary few-shot is closer. LitSpace is a bit better on correctness, completeness, and helpfulness, while summary few-shot is stronger on relevance and faithfulness. A fair read is that LitSpace is better overall, but not by a huge margin on every metric.

## 5. Latency, Token, and Cost Profile

This section looks at the efficiency side of the system using the saved values in `metrics_summary.json`. It compares wall-clock latency, token usage, and estimated API cost across LitSpace and the two baselines.

This matters because LitSpace is doing more than one direct model call. It has to resolve scope, retrieve evidence, assemble context, and then generate an answer. So even if answer quality improves, that improvement comes with extra runtime and token overhead that should be shown clearly.

In [54]:
# Save the latency, token, and cost profile plot used in the README.
def vertical_group_chart(*, title: str, subtitle: str, panels: list[dict], output_path: Path, footnote: str) -> None:
    width = 960
    height = 360
    panel_w = 270
    panel_gap = 30
    chart_top = 96
    chart_h = 160
    bar_w = 44
    bar_gap = 34
    left_start = 42
    body = [
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#f8fafc" />',
        f'<text x="36" y="42" class="title">{escape(title)}</text>',
        f'<text x="36" y="64" class="subtitle">{escape(subtitle)}</text>',
    ]
    for panel_index, panel in enumerate(panels):
        x0 = left_start + panel_index * (panel_w + panel_gap)
        values = [item['value'] for item in panel['items'] if item['value'] is not None]
        max_value = max(values) if values else 1.0
        body.append(f'<text x="{x0 + panel_w / 2:.2f}" y="86" class="label" text-anchor="middle">{escape(panel["label"])}</text>')
        for grid_index in range(5):
            y = chart_top + chart_h - (grid_index * chart_h / 4)
            body.append(f'<line x1="{x0}" y1="{y:.2f}" x2="{x0 + panel_w}" y2="{y:.2f}" stroke="#e2e8f0" stroke-width="1" />')
        for item_index, item in enumerate(panel['items']):
            x = x0 + 22 + item_index * (bar_w + bar_gap)
            value = item['value']
            if value is None:
                bar_h = 0
                y = chart_top + chart_h
                body.append(f'<rect x="{x}" y="{y - 3:.2f}" width="{bar_w}" height="3" fill="#cbd5e1" />')
                value_label = 'N/A'
            else:
                bar_h = 0 if max_value == 0 else (value / max_value) * (chart_h * 0.84)
                y = chart_top + chart_h - bar_h
                body.append(f'<rect x="{x}" y="{y:.2f}" width="{bar_w}" height="{bar_h:.2f}" fill="{item["color"]}" rx="2" />')
                value_label = panel['formatter'](value)
            body.append(f'<text x="{x + bar_w / 2:.2f}" y="{y - 8:.2f}" class="value" text-anchor="middle">{escape(value_label)}</text>')
            label_words = item['label'].split(' ', 1)
            body.append(f'<text x="{x + bar_w / 2:.2f}" y="{chart_top + chart_h + 24}" class="tick" text-anchor="middle">{escape(label_words[0])}</text>')
            if len(label_words) > 1:
                body.append(f'<text x="{x + bar_w / 2:.2f}" y="{chart_top + chart_h + 39}" class="tick" text-anchor="middle">{escape(label_words[1])}</text>')
    body.append(f'<text x="36" y="{height - 28}" class="foot">{escape(footnote)}</text>')
    output_path.write_text(svg_document(width, height, ''.join(body)), encoding='utf-8')

latency_token_cost_plot = PLOTS_DIR / 'latency_token_cost_profile.svg'
system_order = ['litspace_rag', 'zero_shot', 'summary_few_shot']

def total_avg_tokens(system_key: str) -> float | None:
    input_tokens = overall[system_key].get('avg_input_tokens')
    output_tokens = overall[system_key].get('avg_output_tokens')
    if input_tokens is None or output_tokens is None:
        return None
    return input_tokens + output_tokens

vertical_group_chart(
    title='Latency, Token, and Cost Profile',
    subtitle='Values are computed from metrics_summary.json and saved by this notebook.',
    panels=[
        {
            'label': 'Average latency (s)',
            'formatter': lambda value: f'{value:.2f}',
            'items': [
                {'label': SYSTEM_LABELS[key], 'value': overall[key].get('avg_latency_sec'), 'color': SYSTEM_COLORS[key]}
                for key in system_order
            ],
        },
        {
            'label': 'Average tokens / answer',
            'formatter': lambda value: f'{value:.0f}',
            'items': [
                {'label': SYSTEM_LABELS[key], 'value': total_avg_tokens(key), 'color': SYSTEM_COLORS[key]}
                for key in system_order
            ],
        },
        {
            'label': 'Average cost / answer',
            'formatter': lambda value: f'${value:.5f}',
            'items': [
                {'label': SYSTEM_LABELS[key], 'value': overall[key].get('avg_cost_usd'), 'color': SYSTEM_COLORS[key]}
                for key in system_order
            ],
        },
    ],
    output_path=latency_token_cost_plot,
    footnote='Token and cost values come from metrics_summary.json; LitSpace now records backend usage in the evaluation outputs.',
)

print('Saved plot:', latency_token_cost_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/latency_token_cost_profile.svg


In [55]:
# Render the latency, token, and cost table from metrics_summary.json.
latency_rows = []
for system in ['litspace_rag', 'summary_few_shot', 'zero_shot']:
    values = overall[system]
    latency_rows.append([
        SYSTEM_LABELS[system],
        fmt_num(values.get('avg_latency_sec')),
        fmt_num(values.get('latency_ci95')),
        fmt_num(values.get('avg_input_tokens')),
        fmt_num(values.get('avg_output_tokens')),
        fmt_cost(values.get('avg_cost_usd')),
    ])

display_markdown_table(
    ['System', 'Avg latency (s)', 'Latency CI95', 'Avg input tokens', 'Avg output tokens', 'Avg cost / answer'],
    latency_rows,
    '![Latency, token, and cost profile](../results/plots/latency_token_cost_profile.svg)\n\nLitSpace is now the highest-token and highest-cost path because it sends retrieved evidence and grounding instructions to the model, while the baselines use smaller direct prompts or summary cards. Its latency is still close to the prompt-only baselines in this run, which suggests the extra retrieval and context assembly cost is measurable but not dominant.\n\nThe token and cost columns are populated from the current evaluation outputs. They should be interpreted as per-question averages for this benchmark and model configuration, not as universal production costs.',
)


| System | Avg latency (s) | Latency CI95 | Avg input tokens | Avg output tokens | Avg cost / answer |
| --- | ---: | ---: | ---: | ---: | ---: |
| LitSpace RAG | 2.23 | 0.57 | 5242.22 | 256.09 | $0.00508 |
| Summary few-shot | 2.08 | 0.29 | 629.67 | 72.80 | $0.00080 |
| Zero-shot | 2.04 | 0.25 | 150.58 | 92.29 | $0.00053 |

![Latency, token, and cost profile](../results/plots/latency_token_cost_profile.svg)

LitSpace is now the highest-token and highest-cost path because it sends retrieved evidence and grounding instructions to the model, while the baselines use smaller direct prompts or summary cards. Its latency is still close to the prompt-only baselines in this run, which suggests the extra retrieval and context assembly cost is measurable but not dominant.

The token and cost columns are populated from the current evaluation outputs. They should be interpreted as per-question averages for this benchmark and model configuration, not as universal production costs.

## 6. Routing Behavior Metrics

These metrics check whether each system answered, clarified, or refused at the right times. They matter because LitSpace is project-bounded: a good system should not only answer accurately, but should also clarify ambiguous questions and refuse questions outside the uploaded corpus.


In [56]:
# Render routing and scope-control metrics from metrics_summary.json.
routing_metrics = [
    ('Answered rate', 'answered_rate'),
    ('Clarified rate', 'clarified_rate'),
    ('Refused rate', 'refused_rate'),
    ('Over-clarification rate', 'over_clarification_rate'),
    ('Over-refusal rate', 'over_refusal_rate'),
]

display_markdown_table(
    ['Metric', 'LitSpace RAG', 'Summary few-shot', 'Zero-shot'],
    [
        [label] + [fmt_num(overall[system].get(key), 3) for system in ['litspace_rag', 'summary_few_shot', 'zero_shot']]
        for label, key in routing_metrics
    ],
)


| Metric | LitSpace RAG | Summary few-shot | Zero-shot |
| --- | ---: | ---: | ---: |
| Answered rate | 0.711 | 0.778 | 0.756 |
| Clarified rate | 0.156 | 0.133 | 0.244 |
| Refused rate | 0.133 | 0.089 | 0.000 |
| Over-clarification rate | 0.125 | 0.050 | 0.175 |
| Over-refusal rate | 0.073 | 0.000 | 0.000 |

LitSpace answers most benchmark questions, but it also uses clarification and refusal more deliberately than a pure prompt-only system. The over-clarification and over-refusal rates show the remaining routing edge cases: sometimes the system asks for clarification when the benchmark expected an answer, and sometimes it refuses when a more useful grounded answer was possible. These errors connect directly to the limitation discussion because routing mistakes can reduce helpfulness even when retrieval is otherwise working.


## 7. Retrieval Performance

The overall scores show answer quality. This section looks at retrieval directly so it is easier to see where LitSpace is strong and where it is still losing information.

The key split here is paper-level retrieval versus section-level retrieval. Finding the right paper is necessary, but that is not the same as finding the exact section needed to support the answer well.


In [57]:
# Save the retrieval metrics plot for LitSpace.
litspace = overall['litspace_rag']
retrieval_plot = PLOTS_DIR / 'litspace_retrieval_metrics.svg'

horizontal_bar_chart(
    title='LitSpace Retrieval Metrics',
    subtitle='Paper-level retrieval is strong; exact supporting-section retrieval is weaker.',
    items=[
        {'label': 'Paper hit@5', 'value': litspace['paper_hit_at_5_mean'], 'color': '#0f766e'},
        {'label': 'Paper recall@5', 'value': litspace['paper_recall_at_5_mean'], 'color': '#0f766e'},
        {'label': 'Section hit@5', 'value': litspace['section_hit_at_5_mean'], 'color': '#1d4ed8'},
        {'label': 'Section recall@5', 'value': litspace['section_recall_at_5_mean'], 'color': '#1d4ed8'},
    ],
    output_path=retrieval_plot,
    max_value=1.0,
    value_fmt='{:.2f}',
    footnote='All values are proportions between 0 and 1.',
)

print('Saved plot:', retrieval_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/litspace_retrieval_metrics.svg


In [58]:
# Render the LitSpace retrieval metrics table from metrics_summary.json.
retrieval_rows = [
    ['Paper hit@5', fmt_num(litspace.get('paper_hit_at_5_mean'))],
    ['Paper recall@5', fmt_num(litspace.get('paper_recall_at_5_mean'))],
    ['Section hit@5', fmt_num(litspace.get('section_hit_at_5_mean'))],
    ['Section recall@5', fmt_num(litspace.get('section_recall_at_5_mean'))],
    ['Paper MRR@5', fmt_num(litspace.get('paper_mrr_at_5_mean'))],
]

display_markdown_table(
    ['Metric', 'LitSpace'],
    retrieval_rows,
    '![Retrieval metrics](../results/plots/litspace_retrieval_metrics.svg)\n\nPaper retrieval is strong. LitSpace almost always brings in a relevant paper, and the high MRR means the correct paper is usually near the top of the ranking.\n\nSection retrieval is weaker, especially section recall. That means the system often finds the right paper but still misses some of the exact supporting sections it needs. This matters because retrieval is no longer the main bottleneck at the paper level. The weaker part is getting the right evidence inside the paper and then using it cleanly in the answer.',
)


| Metric | LitSpace |
| --- | ---: |
| Paper hit@5 | 0.97 |
| Paper recall@5 | 0.86 |
| Section hit@5 | 0.64 |
| Section recall@5 | 0.31 |
| Paper MRR@5 | 0.95 |

![Retrieval metrics](../results/plots/litspace_retrieval_metrics.svg)

Paper retrieval is strong. LitSpace almost always brings in a relevant paper, and the high MRR means the correct paper is usually near the top of the ranking.

Section retrieval is weaker, especially section recall. That means the system often finds the right paper but still misses some of the exact supporting sections it needs. This matters because retrieval is no longer the main bottleneck at the paper level. The weaker part is getting the right evidence inside the paper and then using it cleanly in the answer.

## 8. Pairwise Comparison Against Baselines

Average scores are useful, but pairwise comparisons answer a different question: if two answers are put side by side, how often does LitSpace win?

This helps show whether the overall advantage is broad across the benchmark or driven by a smaller number of big wins.


In [59]:
# Define the stacked-share chart here because this is the only section that uses it.
def stacked_share_chart(*, title: str, subtitle: str, rows: list[dict], output_path: Path, footnote: str) -> None:
    """Save a stacked proportion chart as SVG."""
    width = 960
    top = 92
    left = 250
    right = 80
    row_h = 64
    chart_h = row_h * len(rows)
    height = top + chart_h + 94
    chart_w = width - left - right
    legend_y = height - 54
    legend_items = [
        ('LitSpace win', '#0f766e'),
        ('Baseline win', '#c2410c'),
        ('Tie', '#94a3b8'),
    ]
    body = [
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#f8fafc" />',
        f'<text x="36" y="42" class="title">{escape(title)}</text>',
        f'<text x="36" y="64" class="subtitle">{escape(subtitle)}</text>',
    ]

    # Add percentage guides so each row reads like a full 0-100% bar.
    for pct in (0, 0.25, 0.5, 0.75, 1.0):
        x = left + pct * chart_w
        body.append(f'<line x1="{x:.2f}" y1="{top - 12}" x2="{x:.2f}" y2="{top + chart_h}" stroke="#e2e8f0" stroke-width="1" />')
        body.append(f'<text x="{x:.2f}" y="{top + chart_h + 22}" class="tick" text-anchor="middle">{int(pct * 100)}%</text>')

    for index, row in enumerate(rows):
        y = top + index * row_h
        bar_y = y + 10
        bar_h = 26
        body.append(f'<text x="{left - 16}" y="{bar_y + 17}" class="label" text-anchor="end">{escape(row["label"])}</text>')
        current_x = left
        for segment in row['segments']:
            segment_w = chart_w * segment['value']
            body.append(f'<rect x="{current_x:.2f}" y="{bar_y}" width="{segment_w:.2f}" height="{bar_h}" fill="{segment["color"]}" />')
            if segment_w >= 68:
                body.append(f'<text x="{current_x + (segment_w / 2):.2f}" y="{bar_y + 17}" class="tick" text-anchor="middle" fill="#ffffff">{segment["label"]}</text>')
            current_x += segment_w

    legend_x = 36
    for label, color in legend_items:
        body.append(f'<rect x="{legend_x}" y="{legend_y}" width="16" height="16" rx="3" fill="{color}" />')
        body.append(f'<text x="{legend_x + 24}" y="{legend_y + 13}" class="tick">{escape(label)}</text>')
        legend_x += 170

    body.append(f'<text x="36" y="{height - 16}" class="foot">{escape(footnote)}</text>')
    output_path.write_text(svg_document(width, height, ''.join(body)), encoding='utf-8')


In [60]:
# Save the pairwise win-rate plot.
pairwise_plot = PLOTS_DIR / 'pairwise_win_rates.svg'

stacked_share_chart(
    title='Pairwise Comparison Against Baselines',
    subtitle='Each row shows LitSpace win rate, baseline win rate, and tie rate.',
    rows=[
        {
            'label': 'LitSpace vs zero-shot',
            'segments': [
                {'label': fmt_pct(pairwise['litspace_vs_zero_shot']['a_win_rate']), 'value': pairwise['litspace_vs_zero_shot']['a_win_rate'], 'color': '#0f766e'},
                {'label': fmt_pct(pairwise['litspace_vs_zero_shot']['b_win_rate']), 'value': pairwise['litspace_vs_zero_shot']['b_win_rate'], 'color': '#c2410c'},
                {'label': fmt_pct(pairwise['litspace_vs_zero_shot']['tie_rate']), 'value': pairwise['litspace_vs_zero_shot']['tie_rate'], 'color': '#94a3b8'},
            ],
        },
        {
            'label': 'LitSpace vs summary few-shot',
            'segments': [
                {'label': fmt_pct(pairwise['litspace_vs_summary_few_shot']['a_win_rate']), 'value': pairwise['litspace_vs_summary_few_shot']['a_win_rate'], 'color': '#0f766e'},
                {'label': fmt_pct(pairwise['litspace_vs_summary_few_shot']['b_win_rate']), 'value': pairwise['litspace_vs_summary_few_shot']['b_win_rate'], 'color': '#c2410c'},
                {'label': fmt_pct(pairwise['litspace_vs_summary_few_shot']['tie_rate']), 'value': pairwise['litspace_vs_summary_few_shot']['tie_rate'], 'color': '#94a3b8'},
            ],
        },
    ],
    output_path=pairwise_plot,
    footnote='Pairwise values are taken directly from metrics_summary.json.',
)

print('Saved plot:', pairwise_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/pairwise_win_rates.svg


In [61]:
# Render pairwise comparison values from metrics_summary.json.
pairwise_rows = [
    ['LitSpace vs zero-shot', fmt_num(pairwise['litspace_vs_zero_shot']['a_win_rate'], 3), fmt_num(pairwise['litspace_vs_zero_shot']['b_win_rate'], 3), fmt_num(pairwise['litspace_vs_zero_shot']['tie_rate'], 3)],
    ['LitSpace vs summary few-shot', fmt_num(pairwise['litspace_vs_summary_few_shot']['a_win_rate'], 3), fmt_num(pairwise['litspace_vs_summary_few_shot']['b_win_rate'], 3), fmt_num(pairwise['litspace_vs_summary_few_shot']['tie_rate'], 3)],
]

display_markdown_table(
    ['Comparison', 'LitSpace win rate', 'Baseline win rate', 'Tie rate'],
    pairwise_rows,
    '![Pairwise comparison](../results/plots/pairwise_win_rates.svg)\n\nThis plot says the same basic thing as the overall table, but in a more direct way. LitSpace beats zero-shot on most examples, which supports the claim that retrieval is helping in a consistent way.\n\nThe summary few-shot comparison is much closer. LitSpace still wins more often than it loses, but the gap is smaller and the tie rate is meaningful. So the result is positive for LitSpace, but it is still a mixed comparison rather than a blowout.',
)


| Comparison | LitSpace win rate | Baseline win rate | Tie rate |
| --- | ---: | ---: | ---: |
| LitSpace vs zero-shot | 0.756 | 0.111 | 0.133 |
| LitSpace vs summary few-shot | 0.489 | 0.267 | 0.244 |

![Pairwise comparison](../results/plots/pairwise_win_rates.svg)

This plot says the same basic thing as the overall table, but in a more direct way. LitSpace beats zero-shot on most examples, which supports the claim that retrieval is helping in a consistent way.

The summary few-shot comparison is much closer. LitSpace still wins more often than it loses, but the gap is smaller and the tie rate is meaningful. So the result is positive for LitSpace, but it is still a mixed comparison rather than a blowout.

## 9. Category-Level Breakdown

The overall averages hide a lot. This section breaks correctness down by question type so it is easier to see where the system is strong and where it is still weaker.

That matters because LitSpace is doing several different tasks, and strong performance in one category does not automatically carry over to another.


In [62]:
# Expand category names so the chart labels read cleanly.
CATEGORY_LABELS = {
    'single_paper_factual': 'Single-paper factual',
    'single_paper_summary': 'Single-paper summary',
    'multi_paper_synthesis': 'Multi-paper synthesis',
    'evidence_lookup': 'Evidence lookup',
    'followup': 'Follow-up',
    'ambiguity': 'Ambiguity',
    'refusal': 'Refusal',
}

category_plot = PLOTS_DIR / 'litspace_category_correctness.svg'

# Use a second color for the weaker categories so they stand out.
horizontal_bar_chart(
    title='LitSpace Correctness by Question Category',
    subtitle='This chart surfaces where LitSpace is strongest and where it struggles.',
    items=[
        {
            'label': CATEGORY_LABELS[category],
            'value': by_category[category]['litspace_rag']['correctness_mean'],
            'color': '#0f766e' if category not in {'ambiguity', 'multi_paper_synthesis'} else '#b45309',
        }
        for category in category_order
    ],
    output_path=category_plot,
    max_value=2.0,
    value_fmt='{:.2f}',
    footnote='Ambiguity and multi-paper synthesis are the clearest weak spots in the current benchmark.',
)

print('Saved plot:', category_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/litspace_category_correctness.svg


In [63]:
# Render the category table from metrics_summary.json.
category_rows = [
    [CATEGORY_LABELS[category], fmt_num(by_category[category]['litspace_rag'].get('correctness_mean'))]
    for category in category_order
]

display_markdown_table(
    ['Category', 'LitSpace correctness'],
    category_rows,
    '![Category breakdown](../results/plots/litspace_category_correctness.svg)\n\nThe strongest categories here are follow-up, evidence lookup, and the single-paper tasks. Those are all settings where the system can stay closer to one paper or one conversation thread, and the scores look solid there.\n\nThe weaker categories are ambiguity and multi-paper synthesis. Both need more than basic retrieval. Ambiguity needs good judgment about when to clarify, and multi-paper synthesis needs cleaner combining of evidence across papers. These are still some of the weaker parts of the system.',
)


| Category | LitSpace correctness |
| --- | ---: |
| Single-paper factual | 1.57 |
| Single-paper summary | 1.71 |
| Multi-paper synthesis | 0.88 |
| Evidence lookup | 1.57 |
| Follow-up | 2.00 |
| Ambiguity | 0.60 |
| Refusal | 1.50 |

![Category breakdown](../results/plots/litspace_category_correctness.svg)

The strongest categories here are follow-up, evidence lookup, and the single-paper tasks. Those are all settings where the system can stay closer to one paper or one conversation thread, and the scores look solid there.

The weaker categories are ambiguity and multi-paper synthesis. Both need more than basic retrieval. Ambiguity needs good judgment about when to clarify, and multi-paper synthesis needs cleaner combining of evidence across papers. These are still some of the weaker parts of the system.

## 10. Failure Analysis

The last section moves from averages to row-level failure labels. The summary metrics show where performance drops, but the failure labels help show what the mistakes actually look like.

That makes this section useful for checking whether the weak categories match the actual error patterns in the saved results.


In [64]:
# Count LitSpace failures from the row-level error file.
# Skip rows labeled "good" so the chart stays focused on actual failures.
failure_counts = Counter()
with ERROR_CSV.open('r', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        if row['system_name'] != 'litspace_rag':
            continue
        failure_type = row['failure_type'].strip()
        if failure_type and failure_type != 'good':
            failure_counts[failure_type] += 1

# Sort failures from most common to least common before plotting.
failure_items = [
    {'label': failure_type.replace('_', ' '), 'value': count, 'color': '#c2410c'}
    for failure_type, count in sorted(failure_counts.items(), key=lambda item: (-item[1], item[0]))
]

failure_plot = PLOTS_DIR / 'litspace_failure_types.svg'

horizontal_bar_chart(
    title='LitSpace Failure Types',
    subtitle='Counts are aggregated from evaluation/results/error_analysis.csv for LitSpace rows only.',
    items=failure_items,
    output_path=failure_plot,
    max_value=max(item['value'] for item in failure_items) if failure_items else 1,
    value_fmt='{:.0f}',
    footnote='The largest buckets are unsupported-claim cases, missing-key-point cases, and clarification-related failures.',
)

print('Saved plot:', failure_plot)
failure_counts


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/litspace_failure_types.svg


Counter({'unsupported_claim': 14,
         'missing_key_point': 11,
         'over_clarification': 4,
         'wrong_paper': 3,
         'should_have_clarified': 3,
         'over_refusal': 1,
         'should_have_refused': 1})

In [65]:
# Render failure counts from evaluation/results/error_analysis.csv.
display_markdown_table(
    ['Failure type', 'Count'],
    [[item['label'].title(), str(item['value'])] for item in failure_items],
    '![Failure types](../results/plots/litspace_failure_types.svg)\n\nThe two biggest failure types are unsupported claims and missing key points. That suggests a lot of the mistakes are partial-answer mistakes, not total breakdowns. The system is often in the right area, but the final answer is either not grounded enough or not complete enough.\n\nThe clarification-related failures line up with the weak ambiguity score from the category chart. LitSpace sometimes asks for clarification when it does not need to, and sometimes skips clarification when it probably should. So ambiguity is still a clear place where the system needs more work.',
)


| Failure type | Count |
| --- | ---: |
| Unsupported Claim | 14 |
| Missing Key Point | 11 |
| Over Clarification | 4 |
| Should Have Clarified | 3 |
| Wrong Paper | 3 |
| Over Refusal | 1 |
| Should Have Refused | 1 |

![Failure types](../results/plots/litspace_failure_types.svg)

The two biggest failure types are unsupported claims and missing key points. That suggests a lot of the mistakes are partial-answer mistakes, not total breakdowns. The system is often in the right area, but the final answer is either not grounded enough or not complete enough.

The clarification-related failures line up with the weak ambiguity score from the category chart. LitSpace sometimes asks for clarification when it does not need to, and sometimes skips clarification when it probably should. So ambiguity is still a clear place where the system needs more work.

## 11. Quick Demo Notes

`demo/quick_demo.py` is the main demo entry point for the repo.

- by default, it starts the local backend if it is not already running
- it creates a fresh project from `demo/corpus/`
- it uploads the 7 demo PDFs, parses them, chunks them, and indexes them
- it creates a chat and asks 3 sample questions through the real backend
- the backend handles provider fallback automatically if no GPT key is available
- `--static` prints committed answers from `evaluation/outputs/litspace_outputs.jsonl`

The main path is intentionally simple: run one command and get a real demo project plus three example questions.
